# 🔄 使用 Microsoft Foundry 的基本代理工作流程 (Python)

## 📋 工作流程編排教學

本筆記本介紹了 Microsoft 代理框架強大的 <strong>工作流程建構器</strong> 功能。學習如何創建複雜的多步代理工作流程，以便處理複雜的業務流程並無縫協調多個 AI 操作。

> **遷移說明：** 此範例之前參考 GitHub 模型。GitHub 模型將於 2026 年 7 月退役，因此現在使用透過 `FoundryChatClient` 的 **Microsoft Foundry** ，其目標為 Azure OpenAI **Responses API**。

## 🎯 學習目標

### 🏗️ <strong>工作流程架構</strong>
- <strong>工作流程建構器</strong>：設計和編排複雜的多步驟流程
- <strong>事件驅動執行</strong>：處理工作流程事件和狀態轉換
- <strong>視覺化工作流程設計</strong>：創建並可視化工作流程結構
- **Microsoft Foundry 整合**：在工作流程上下文中運用 AI 模型

### 🔄 <strong>流程編排</strong>
- <strong>順序操作</strong>：按邏輯順序串連多個代理任務
- <strong>條件邏輯</strong>：實現決策點和分支工作流程
- <strong>錯誤處理</strong>：強健的錯誤恢復與工作流程韌性
- <strong>狀態管理</strong>：追蹤和管理工作流程執行狀態

### 📊 <strong>企業工作流程範式</strong>
- <strong>業務流程自動化</strong>：自動化複雜的組織工作流程
- <strong>多代理協調</strong>：協調多個專門代理
- <strong>可擴展執行</strong>：設計適用於企業規模操作的工作流程
- <strong>監控與可觀察性</strong>：追蹤工作流程效能及結果

## ⚙️ 前置條件與設置

### 📦 <strong>必要依賴</strong>

安裝具備工作流程功能的代理框架：

```bash
pip install agent-framework -U
```

### 🔑 **Microsoft Foundry 配置**

先用 Azure CLI (`az login`) 登入，讓 `AzureCliCredential` 進行驗證，然後設定你的 Microsoft Foundry 專案詳細資料。

**環境設置 (.env 檔)：**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

### 🏢 <strong>企業使用案例</strong>

**業務流程範例：**
- <strong>客戶入職</strong>：多步驟驗證與設定工作流程
- <strong>內容管道</strong>：自動內容創建、審核與發布
- <strong>數據處理</strong>：搭配 AI 變換的 ETL 工作流程
- <strong>品質保證</strong>：自動化測試與驗證流程

**工作流程優勢：**
- 🎯 <strong>可靠性</strong>：確定性執行與錯誤恢復
- 📈 <strong>可擴展性</strong>：處理高量級流程自動化
- 🔍 <strong>可觀察性</strong>：完整稽核追蹤與監控
- 🔧 <strong>維護性</strong>：視覺化設計與模組化元件

## 🎨 工作流程設計範式

### 基本工作流程結構
```mermaid
graph TD
    A[開始] --> B[代理任務 1]
    B --> C{決策點}
    C -->|成功| D[代理任務 2]
    C -->|失敗| E[錯誤處理器]
    D --> F[結束]
    E --> F
```

**主要組件：**
- **WorkflowBuilder**：主要編排引擎
- **WorkflowEvent**：事件處理與通信
- **WorkflowViz**：視覺化工作流程表示與除錯

讓我們一起建立你的第一個智能工作流程！🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Microsoft Foundry project settings from .env file
load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = provider.as_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = provider.as_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
